# TSM y Anomalía de TSM — datos tal cual como en ENSO_DASHBOARD

Notebook standalone para Google Colab que se conecta a la **misma base de datos** que usa el pipeline del proyecto (`pipeline/fetch_and_render.py` + `app/plotting.py`), replicando exactamente:

- Las URLs de descarga (NOAA PSL, dataset OISST v2 high-res).
- El recorte de región al Pacífico tropical (`LON_RANGE = (115, 330)`, `LAT_RANGE = (-25, 25)`).
- La paleta y las cajas Niño (mismos colores/límites que el dashboard).
- El GIF de los últimos 15 días (mismo patrón que `build_recent_animation()` del pipeline).

Orden del notebook:
1. Setup (instalación, descarga, recorte).
2. **Último dato disponible — TSM.**
3. **Último dato disponible — Anomalía de TSM.**
4. **GIF de los últimos 15 días — TSM** (por separado).
5. **GIF de los últimos 15 días — Anomalía de TSM** (por separado).

Fuente de datos:
- Página del dataset: https://psl.noaa.gov/data/gridded/data.noaa.oisst.v2.highres.html
- Catálogo THREDDS: https://psl.noaa.gov/thredds/catalog/Datasets/noaa.oisst.v2.highres/catalog.html

No requiere ningún archivo del VPS — descarga todo directo desde NOAA al runtime de Colab.

## 0. Setup

In [ ]:
# Dependencias (cartopy e imageio no vienen preinstalados en Colab)
!pip -q install xarray netCDF4 cftime cartopy imageio

In [ ]:
import os
import glob
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from PIL import Image
import imageio.v2 as imageio
from IPython.display import Image as IPImage, display

# ── Mismas constantes que pipeline/fetch_and_render.py ──────────────────
YEAR = datetime.now(timezone.utc).year
NOAA_BASE = "https://downloads.psl.noaa.gov/Datasets"
SOURCES = {
    "sst_mean": f"{NOAA_BASE}/noaa.oisst.v2.highres/sst.day.mean.{YEAR}.nc",
    "sst_anom": f"{NOAA_BASE}/noaa.oisst.v2.highres/sst.day.anom.{YEAR}.nc",
}

LON_RANGE = (115, 330)   # Pacífico tropical, convención 0-360
LAT_RANGE = (-25, 25)

print(SOURCES)

In [ ]:
# ── Descarga a disco local del runtime de Colab ──────────────────────────
os.makedirs("raw", exist_ok=True)
os.makedirs("frames", exist_ok=True)

for name, url in SOURCES.items():
    dest = f"raw/{name}.nc"
    if not os.path.exists(dest):
        print(f"Descargando {name} ...")
        !wget -q -O {dest} "{url}"
    else:
        print(f"{name} ya está descargado")

!ls -lh raw/

In [ ]:
# ── subset_var(): misma función que pipeline/fetch_and_render.py ────────
def subset_var(da, lon_range=LON_RANGE, lat_range=LAT_RANGE):
    """Recorta lon/lat sin asumir el orden (ascendente o descendente) del eje
    lat, tal cual el proyecto original."""
    lat_vals = da["lat"].values
    lat_sl = slice(*lat_range) if lat_vals[0] < lat_vals[-1] else slice(*reversed(lat_range))
    return da.sel(lon=slice(*lon_range), lat=lat_sl)


sst_mean = subset_var(xr.open_dataset("raw/sst_mean.nc")["sst"])
sst_anom = subset_var(xr.open_dataset("raw/sst_anom.nc")["anom"])

ultima_fecha_tsm = str(sst_mean.time.values[-1])[:10]
ultima_fecha_anom = str(sst_anom.time.values[-1])[:10]

print("TSM:", sst_mean.dims, sst_mean.shape, "| última fecha:", ultima_fecha_tsm)
print("Anomalía TSM:", sst_anom.dims, sst_anom.shape, "| última fecha:", ultima_fecha_anom)

In [ ]:
# ── Mismas constantes/funciones que app/plotting.py ──────────────────────
EXTENT_PACIFICO = [115, 360 - 30, -21, 21]

NINO_COLORS = {
    "Niño 4":   "#7b1fa2",
    "Niño 3.4": "#1565c0",
    "Niño 3":   "#2e7d32",
    "Niño 1+2": "#c62828",
}

NINO_BOXES = {
    "Niño 3.4": dict(xy=(-170, -5), width=50, height=10, label_xy=(-150, 7)),
    "Niño 1+2": dict(xy=(-90, -10), width=10, height=10, label_xy=(-90, 2)),
    "Niño 3":   dict(xy=(-150, -5), width=60, height=10, label_xy=(-120, 7)),
    "Niño 4":   dict(xy=(160, -5), width=50, height=10, label_xy=(180, 7)),
}


def _add_nino_boxes(ax):
    for name, b in NINO_BOXES.items():
        color = NINO_COLORS[name]
        ax.add_patch(mpatches.Rectangle(
            xy=b["xy"], width=b["width"], height=b["height"],
            facecolor=color, edgecolor="none", alpha=0.12,
            transform=ccrs.PlateCarree()))
        ax.add_patch(mpatches.Rectangle(
            xy=b["xy"], width=b["width"], height=b["height"],
            facecolor="none", edgecolor=color, linewidth=2,
            transform=ccrs.PlateCarree()))
        ax.text(*b["label_xy"], name, color=color, fontsize=9, fontweight="bold",
                transform=ccrs.PlateCarree())


def _base_map(figsize=(12, 6)):
    fig, ax = plt.subplots(1, 1, figsize=figsize,
                            subplot_kw={"projection": ccrs.PlateCarree(central_longitude=180)})
    ax.set_extent(EXTENT_PACIFICO, crs=ccrs.PlateCarree())
    ax.coastlines(resolution="50m")
    ax.add_feature(cfeature.BORDERS, linestyle=":", edgecolor="black")
    ax.add_feature(cfeature.LAND, edgecolor="black", facecolor="lightgray")
    ax.add_feature(cfeature.OCEAN)
    gl = ax.gridlines(draw_labels=True, x_inline=False, y_inline=False,
                       color="gray", linestyle="--", alpha=0.6,
                       xlocs=np.arange(-180, 180, 20), ylocs=np.arange(-90, 91, 5))
    gl.top_labels = gl.right_labels = False
    fig.subplots_adjust(left=0.06, right=0.90, top=0.90, bottom=0.12)
    return fig, ax


def _autocrop_whitespace(path, pad=14, max_gap=40):
    """Mismo recorte de margen blanco que app/plotting.py — evita que cada
    frame del GIF quede con un borde/tamaño ligeramente distinto."""
    im = Image.open(path).convert("RGB")
    arr = np.asarray(im)
    non_white = np.any(arr < 250, axis=2)
    rows = np.where(non_white.any(axis=1))[0]
    cols = np.where(non_white.any(axis=0))[0]
    if rows.size == 0 or cols.size == 0:
        return
    top, bottom = max(int(rows.min()) - pad, 0), min(int(rows.max()) + pad, arr.shape[0])
    left, right = max(int(cols.min()) - pad, 0), min(int(cols.max()) + pad, arr.shape[1])
    arr = arr[top:bottom, left:right]

    row_has_content = np.any(arr < 250, axis=(1, 2))
    keep = np.ones(len(row_has_content), dtype=bool)
    i = 0
    while i < len(row_has_content):
        if row_has_content[i]:
            i += 1
            continue
        j = i
        while j < len(row_has_content) and not row_has_content[j]:
            j += 1
        if j - i > max_gap:
            keep[i + max_gap:j] = False
        i = j
    Image.fromarray(arr[keep]).save(path)


def plot_sst(sst_daily_data, date_str, out_path=None, show=True,
             min_val=14, max_val=32, levels=19):
    target_date = pd.to_datetime(date_str)
    sst_to_plot = sst_daily_data.sel(time=target_date, method="nearest").squeeze()

    fig, ax = _base_map()
    cmap = plt.get_cmap("Spectral_r")
    contour_levels = np.linspace(min_val, max_val, levels)
    plot_obj = sst_to_plot.plot.contourf(ax=ax, transform=ccrs.PlateCarree(), cmap=cmap,
                                          levels=contour_levels, add_colorbar=False, extend="both")
    fig.suptitle(f'TSM Diaria {target_date.strftime("%Y-%m-%d")}', fontsize=14, fontweight="bold")
    _add_nino_boxes(ax)
    cbar_ax = fig.add_axes([0.92, 0.35, 0.02, 0.3])
    cbar = fig.colorbar(plot_obj, cax=cbar_ax, orientation="vertical")
    cbar.set_label("TSM (°C)", fontsize=12, fontweight="bold")

    if out_path:
        fig.savefig(out_path, dpi=150)
        _autocrop_whitespace(out_path)
    if show:
        plt.show()
    else:
        plt.close(fig)
    return out_path


def plot_sst_anom(sst_anom_data, date_str, out_path=None, show=True,
                   min_val=-5, max_val=5, levels=21):
    target_date = pd.to_datetime(date_str)
    sst_to_plot = sst_anom_data.sel(time=target_date, method="nearest").squeeze()

    fig, ax = _base_map()
    cmap = plt.get_cmap("seismic")
    contour_levels = np.linspace(min_val, max_val, levels)
    plot_obj = sst_to_plot.plot.contourf(ax=ax, transform=ccrs.PlateCarree(), cmap=cmap,
                                          levels=contour_levels, add_colorbar=False, extend="both")
    fig.suptitle(f'Anomalía TSM Diaria {target_date.strftime("%Y-%m-%d")}', fontsize=14, fontweight="bold")
    _add_nino_boxes(ax)
    cbar_ax = fig.add_axes([0.92, 0.35, 0.02, 0.3])
    cbar = fig.colorbar(plot_obj, cax=cbar_ax, orientation="vertical")
    cbar.set_label("Anom TSM (°C)", fontsize=12, fontweight="bold")

    if out_path:
        fig.savefig(out_path, dpi=150)
        _autocrop_whitespace(out_path)
    if show:
        plt.show()
    else:
        plt.close(fig)
    return out_path

## 1. Último dato disponible — TSM

In [ ]:
print("Último dato de TSM:", ultima_fecha_tsm)
plot_sst(sst_mean, ultima_fecha_tsm)

## 2. Último dato disponible — Anomalía de TSM

In [ ]:
print("Último dato de anomalía TSM:", ultima_fecha_anom)
plot_sst_anom(sst_anom, ultima_fecha_anom)

## 3. GIF de los últimos 15 días — TSM

Mismo patrón que `build_recent_animation()` del pipeline: un frame por día (últimos 15 días con dato disponible), mismo tamaño de canvas, `duration=500ms`, loop infinito.

In [ ]:
def build_gif(da, plot_fn, prefix, max_frames=15, duration_ms=500):
    fechas = pd.to_datetime(da["time"].values)
    fechas_str = sorted({d.strftime("%Y-%m-%d") for d in fechas})[-max_frames:]

    frame_paths = []
    for fecha in fechas_str:
        out_path = f"frames/{prefix}_{fecha}.png"
        plot_fn(da, fecha, out_path=out_path, show=False)
        frame_paths.append(out_path)

    frames = [imageio.imread(fp) for fp in frame_paths]
    max_h = max(f.shape[0] for f in frames)
    max_w = max(f.shape[1] for f in frames)
    padded = []
    for f in frames:
        h, w = f.shape[:2]
        if (h, w) == (max_h, max_w):
            padded.append(f)
            continue
        canvas = np.full((max_h, max_w, f.shape[2]), 255, dtype=f.dtype)
        canvas[:h, :w] = f
        padded.append(canvas)

    gif_path = f"{prefix}_anim.gif"
    imageio.mimsave(gif_path, padded, duration=duration_ms, loop=0)
    print(f"GIF armado con {len(frame_paths)} días: {gif_path}")
    return gif_path


gif_tsm = build_gif(sst_mean, plot_sst, "tsm")
display(IPImage(filename=gif_tsm))

## 4. GIF de los últimos 15 días — Anomalía de TSM

In [ ]:
gif_anom = build_gif(sst_anom, plot_sst_anom, "anom")
display(IPImage(filename=gif_anom))